<a href="https://colab.research.google.com/github/GuddetiHarun/GroupDNA-your-WhatsApp-group-chat-decoded/blob/main/GroupDNA_%3CHarun%3E_%3C21921%3E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

GroupDNA
your WhatsApp group chat, decoded.
"Spotify Wrapped, but for your friend group."

Guddeti Harun Kumar Reddy

Roll no:21921

Btach:september

In [ ]:
import numpy as np
from datetime import datetime, timedelta

FILE_PATH = '/content/hostel_bois.txt'

try:
    open(FILE_PATH, 'r', encoding='utf-8').close()
except FileNotFoundError:
    FILE_PATH = 'hostel_bois.txt'

Feature 1:Parser

In [ ]:
def is_message_start(line):
    """A new chat entry starts with a date like 12/04/24 (first 8 characters = NN/NN/NN)."""
    if len(line) < 8:
        return False
    return (line[2] == '/' and line[5] == '/' and
            line[0:2].isdigit() and line[3:5].isdigit() and line[6:8].isdigit())


def parse_chat(path):
    """Return (messages, system_count).
    Each message is a dict: timestamp, sender, text, kind ('text' | 'media' | 'deleted'), dt (datetime).
    Media / deleted messages are kept (the person DID send something) but tagged so
    word and length stats can skip them."""
    with open(path, 'r', encoding='utf-8') as f:
        lines = f.read().split('\n')

    messages = []
    system_count = 0
    last_was_real = False

    for raw in lines:
        line = raw.strip('\r')
        if line.strip() == '':
            continue

        if not is_message_start(line):
            if last_was_real:
                messages[-1]['text'] += ' ' + line.strip()
            continue

        if ' - ' not in line:
            system_count += 1
            last_was_real = False
            continue

        stamp, rest = line.split(' - ', 1)
        if ': ' not in rest:
            system_count += 1
            last_was_real = False
            continue

        sender, text = rest.split(': ', 1)
        text = text.strip()

        kind = 'text'
        if text == '<Media omitted>':
            kind = 'media'
        elif text == 'This message was deleted':
            kind = 'deleted'

        messages.append({
            'timestamp': stamp,
            'sender': sender.strip(),
            'text': text,
            'kind': kind,
            'dt': datetime.strptime(stamp, '%d/%m/%y, %H:%M'),
        })
        last_was_real = True

    return messages, system_count


messages, system_count = parse_chat(FILE_PATH)
HAS_DATA = len(messages) > 0

if not HAS_DATA:
    print('No messages found in the file. Nothing to analyse.')
else:
    people = sorted(set(m['sender'] for m in messages))
    text_messages = [m for m in messages if m['kind'] == 'text']
    media_count, deleted_count = {}, {}
    for m in messages:
        if m['kind'] == 'media':
            media_count[m['sender']] = media_count.get(m['sender'], 0) + 1
        elif m['kind'] == 'deleted':
            deleted_count[m['sender']] = deleted_count.get(m['sender'], 0) + 1

    first_day = messages[0]['dt'].replace(hour=0, minute=0)
    last_day = messages[-1]['dt'].replace(hour=0, minute=0)
    total_days = (last_day - first_day).days + 1

    print(f"Successfully parsed {len(messages)} messages from {len(people)} participants over "
          f"{total_days} days, skipped {system_count} system messages, "
          f"{sum(media_count.values())} media-omitted, {sum(deleted_count.values())} deleted messages.")
    print('\nFirst 5:')
    for m in messages[:5]:
        print('  ', m['timestamp'], '|', m['sender'], '|', m['text'][:50])
    print('Last 5:')
    for m in messages[-5:]:
        print('  ', m['timestamp'], '|', m['sender'], '|', m['text'][:50])

Successfully parsed 3174 messages from 6 participants over 60 days, skipped 4 system messages, 32 media-omitted, 15 deleted messages.

First 5:
   01/04/24, 01:17 | Rahul | scene fix
   01/04/24, 01:17 | Rahul | haan
   01/04/24, 01:18 | Rahul | kya scene
   01/04/24, 02:13 | Rahul | abhi free hai?
   01/04/24, 02:13 | Rahul | abey
Last 5:
   30/05/24, 19:14 | Priya | Take care everyone
   30/05/24, 19:28 | Priya | Karan that sounds tough, take care
   30/05/24, 21:17 | Aman | the existential dread is back
   30/05/24, 21:30 | Karan | Long day guys, woke up at six for that placement w
   30/05/24, 23:31 | Aman | anyone awake?


Feature 2:Group Overview

In [ ]:
def message_counts(messages):
    """{person: number of messages}"""
    counts = {}
    for m in messages:
        counts[m['sender']] = counts.get(m['sender'], 0) + 1
    return counts


def word_and_length_stats(text_messages):
    """{person: (total words, average words per message)} - real text only."""
    words, msgs = {}, {}
    for m in text_messages:
        n = len(m['text'].split())
        words[m['sender']] = words.get(m['sender'], 0) + n
        msgs[m['sender']] = msgs.get(m['sender'], 0) + 1
    return {p: (words[p], words[p] / msgs[p]) for p in words}


if HAS_DATA:
    per_person = message_counts(messages)
    ranking = sorted(per_person.items(), key=lambda kv: kv[1], reverse=True)
    stats = word_and_length_stats(text_messages)
    for name, n in ranking:
        w = stats.get(name, (0, 0))
        print(f"{name:<8} {n:>5} msgs   {w[0]:>6} words   {w[1]:>5.1f} words/msg")

Rahul      953 msgs     2399 words     2.6 words/msg
Priya      718 msgs     3560 words     5.0 words/msg
Neha       635 msgs     3317 words     5.3 words/msg
Aman       490 msgs     2430 words     5.0 words/msg
Karan      354 msgs    19681 words    57.0 words/msg
Vikas       24 msgs       40 words     1.8 words/msg


Feature 3: Most Active Day and Hour

In [ ]:
def busiest_day_and_hour(messages):
    day_counts, hour_counts = {}, {}
    for m in messages:
        day = m['dt'].strftime('%d %B %Y')
        hour = m['dt'].hour
        day_counts[day] = day_counts.get(day, 0) + 1
        hour_counts[hour] = hour_counts.get(hour, 0) + 1
    best_day = max(day_counts, key=day_counts.get)
    best_hour = max(hour_counts, key=hour_counts.get)
    return best_day, day_counts[best_day], best_hour, hour_counts[best_hour]


if HAS_DATA:
    bd, bd_n, bh, bh_n = busiest_day_and_hour(messages)
    print(f"Busiest day  : {bd} ({bd_n} messages)")
    print(f"Busiest hour : {bh:02d}:00 - {bh + 1:02d}:00 (avg {bh_n / total_days:.0f} messages per day)")

Busiest day  : 04 May 2024 (76 messages)
Busiest hour : 18:00 - 19:00 (avg 4 messages per day)


Feature 4: activity Heatmap

In [ ]:
def build_heatmap(messages, people):
    """rows = people (same order as `people`), columns = hour 0..23."""
    matrix = np.zeros((len(people), 24), dtype=int)
    for m in messages:
        matrix[people.index(m['sender']), m['dt'].hour] += 1
    return matrix


def shade(value, row_max):
    """4 shading levels relative to that person's own busiest hour."""
    if row_max == 0 or value == 0:
        return '. '
    ratio = value / row_max
    if ratio <= 0.25:
        return '. '
    if ratio <= 0.50:
        return '\u2591 '
    if ratio <= 0.75:
        return '\u2592 '
    return '\u2588 '


def heatmap_row(counts_row):
    row_max = counts_row.max()
    return ''.join(shade(v, row_max) for v in counts_row)


if HAS_DATA:
    heat = build_heatmap(messages, people)
    print('Matrix shape:', heat.shape)
    print('Row totals  :', dict(zip(people, heat.sum(axis=1).tolist())))
    print('Busiest hour overall (column sums):', int(heat.sum(axis=0).argmax()))
    header = ''.join(f"{h:02d}    " if h % 3 == 0 else '' for h in range(24))
    print('\n        ' + header)
    for i, p in enumerate(people):
        print(f"{p:<8}{heatmap_row(heat[i])}")

Matrix shape: (6, 24)
Row totals  : {'Aman': 490, 'Karan': 354, 'Neha': 635, 'Priya': 718, 'Rahul': 953, 'Vikas': 24}
Busiest hour overall (column sums): 18

        00    03    06    09    12    15    18    21    
Aman    ▒ █ ▒ ▒ █ . . . . . . . . . . . . . . . . . . ▒ 
Karan   . . . . . . . . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . . 
Neha    . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░ 
Priya   . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ . 
Rahul   . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒ 
Vikas   . . . . . . . ░ █ ░ ░ . ▒ ▒ . ░ ░ █ ▒ ▒ ░ ░ ░ ▒ 


Feature 5:Top Words

In [ ]:
STOP_WORDS = set([
    'i', 'is', 'the', 'a', 'an', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'at', 'it', 'its',
    'was', 'were', 'be', 'been', 'am', 'are', 'this', 'that', 'these', 'those', 'so', 'my', 'me',
    'you', 'your', 'he', 'his', 'she', 'her', 'we', 'they', 'them', 'our', 'us', 'with', 'as',
    'by', 'from', 'up', 'do', 'did', 'does', 'have', 'has', 'had', 'not', 'no', 'but', 'if',
    'then', 'than', 'too', 'just', 'about', 'which', 'what', 'how', 'when', 'who', 'will', 'would',
    'can', 'could', 'one', 'all', 'out', 'there', 'their', 'into', 'over', 'also', 'like',
    'im', "i'm", "it's", "that's", "don't", 'now',
    'hai', 'ho', 'hain', 'tha', 'thi', 'ka', 'ki', 'ke', 'ko', 'se', 'ne', 'me', 'aur', 'toh', 'to',
])
PUNCTUATION = '.,!?;:"()[]{}<>-_*/\\~`|'


def tokenize(text):
    """lowercase -> split -> strip punctuation -> drop stop words / empty / pure numbers."""
    words = []
    for raw in text.lower().split():
        w = raw.strip(PUNCTUATION)
        if w == '' or w in STOP_WORDS or w.isdigit():
            continue
        words.append(w)
    return words


def word_frequencies(msgs):
    freq = {}
    for m in msgs:
        for w in tokenize(m['text']):
            freq[w] = freq.get(w, 0) + 1
    return freq


def top_n(freq, n):
    return sorted(freq.items(), key=lambda kv: kv[1], reverse=True)[:n]


def bar(value, max_value, width=20):
    n = int(round(width * value / max_value)) if max_value else 0
    return '\u2588' * n if n > 0 else '.'


if HAS_DATA:
    group_top = top_n(word_frequencies(text_messages), 10)
    for w, c in group_top:
        print(f"{w:<10} {bar(c, group_top[0][1]):<20} {c}")
    print('\nTop 5 words per person:')
    for p in people:
        mine = [m for m in text_messages if m['sender'] == p]
        print(f"  {p:<8}", ', '.join(f"{w}({c})" for w, c in top_n(word_frequencies(mine), 5)))

guys       ████████████████████ 318
today      ████████████████     257
everyone   ████████████         187
telling    ███████████          179
bhai       ██████████           160
started    █████████            150
scene      █████████            145
entire     █████████            145
please     █████████            141
anyone     █████████            139

Top 5 words per person:
  Aman     sleep(71), anyone(49), wonder(43), night(41), can't(34)
  Karan    telling(179), today(171), started(150), entire(145), guys(120)
  Neha     guys(101), cant(58), ok(52), today(48), way(48)
  Priya    please(141), everyone(137), aman(93), anyone(90), okay(80)
  Rahul    bhai(159), scene(144), kya(133), yaar(105), ja(101)
  Vikas    haha(4), sorry(3), busy(3), haan(3), exam(2)


Feature 6: Response Speed & Silent Streaks

In [ ]:
def average_response_times(messages, people):
    """For every message sent by someone else, measure the gap until THIS person's next message.
    Walk the chat backwards, remembering each person's next message time."""
    next_time = {p: None for p in people}
    gaps = {p: [] for p in people}
    for m in reversed(messages):
        for p in people:
            if p != m['sender'] and next_time[p] is not None:
                gaps[p].append((next_time[p] - m['dt']).total_seconds())
        next_time[m['sender']] = m['dt']
    return {p: (sum(g) / len(g) if g else None) for p, g in gaps.items()}


def pretty_duration(seconds):
    if seconds is None:
        return 'n/a'
    if seconds < 3600:
        return f"{seconds / 60:.1f} minutes"
    return f"{seconds / 3600:.1f} hours"


def active_days_per_person(messages, people):
    days = {p: set() for p in people}
    for m in messages:
        days[m['sender']].add(m['dt'].date())
    return days


def longest_silent_streaks(active_days, people, first_day, total_days):
    """{person: (streak_length, start_date, end_date)} - longest run of days with zero messages."""
    all_days = [(first_day + timedelta(days=i)).date() for i in range(total_days)]
    result = {}
    for p in people:
        best, best_start, best_end = 0, None, None
        cur, cur_start = 0, None
        for d in all_days:
            if d in active_days[p]:
                cur = 0
            else:
                if cur == 0:
                    cur_start = d
                cur += 1
                if cur > best:
                    best, best_start, best_end = cur, cur_start, d
        result[p] = (best, best_start, best_end)
    return result


if HAS_DATA:
    avg_gap = average_response_times(messages, people)
    active_days = active_days_per_person(messages, people)
    streaks = longest_silent_streaks(active_days, people, first_day, total_days)
    for p, v in sorted(avg_gap.items(), key=lambda kv: (kv[1] is None, kv[1])):
        print(f"{p:<8} avg response: {pretty_duration(v)}")
    print()
    for p, (n, s, e) in sorted(streaks.items(), key=lambda kv: kv[1][0], reverse=True):
        extra = f"  ({s.strftime('%d %b')} - {e.strftime('%d %b')})" if n else ''
        print(f"{p:<8} longest silent streak: {n} days{extra}")

Priya    avg response: 3.1 hours
Karan    avg response: 4.7 hours
Aman     avg response: 4.8 hours
Neha     avg response: 5.1 hours
Rahul    avg response: 5.5 hours
Vikas    avg response: 79.4 hours

Vikas    longest silent streak: 11 days  (23 Apr - 03 May)
Aman     longest silent streak: 0 days
Karan    longest silent streak: 0 days
Neha     longest silent streak: 0 days
Priya    longest silent streak: 0 days
Rahul    longest silent streak: 0 days


Feature 7:Personality Archtypes

In [ ]:

CARING_WORDS = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please',
                'reminder', 'drink water', "don't forget"]
LAUGH_WORDS = ['lol', 'lmao', 'haha', 'rofl', 'lmfao']

PANDIT_WORDS = ['placement', 'internship', 'resume', 'interview', 'workshop', 'deadline',
                'assignment', 'project', 'exam', 'viva', 'attendance']


def msgs_of(person, msgs):
    return [m for m in msgs if m['sender'] == person]


def pct(part, whole):
    return 100 * part / whole if whole else 0


def score_spammer(person, ctx):
    """Average length of runs of back-to-back messages with nobody else speaking."""
    bursts, run, prev = [], 0, None
    for m in ctx['all']:
        if m['sender'] == person:
            run = run + 1 if prev == person else 1
        else:
            if prev == person and run:
                bursts.append(run)
            run = 0
        prev = m['sender']
    if prev == person and run:
        bursts.append(run)
    return (sum(bursts) / len(bursts) if bursts else 0), 3.0


def score_group_mom(person, ctx):
    """% of messages containing at least one caring keyword. Threshold: 30%."""
    mine = msgs_of(person, ctx['text'])
    hits = sum(1 for m in mine if any(k in m['text'].lower() for k in CARING_WORDS))
    return pct(hits, len(mine)), 30.0


def score_night_owl(person, ctx):
    """% of messages sent between 23:00 and 04:59."""
    mine = msgs_of(person, ctx['all'])
    night = sum(1 for m in mine if m['dt'].hour >= 23 or m['dt'].hour <= 4)
    return pct(night, len(mine)), 60.0


def score_storyteller(person, ctx):
    """Average words per (real text) message."""
    mine = msgs_of(person, ctx['text'])
    words = sum(len(m['text'].split()) for m in mine)
    return (words / len(mine) if mine else 0), 30.0


def score_drama_queen(person, ctx):
    """% of messages that are ALL CAPS (>= 3 chars) OR contain 2+ exclamation marks."""
    mine = msgs_of(person, ctx['text'])
    loud = sum(1 for m in mine
               if (m['text'].isupper() and len(m['text']) >= 3) or m['text'].count('!') >= 2)
    return pct(loud, len(mine)), 30.0


def score_ghost(person, ctx):
    """% of days in the chat period with zero messages."""
    return 100 - pct(len(ctx['active_days'][person]), ctx['total_days']), 60.0


def score_comedian(person, ctx):
    """% of messages with a laugh word. Own threshold: 25%."""
    mine = msgs_of(person, ctx['text'])
    hits = sum(1 for m in mine if any(k in m['text'].lower() for k in LAUGH_WORDS))
    return pct(hits, len(mine)), 25.0


def score_question_master(person, ctx):
    """% of messages ending with '?'."""
    mine = msgs_of(person, ctx['text'])
    return pct(sum(1 for m in mine if m['text'].endswith('?')), len(mine)), 25.0


def score_placement_pandit(person, ctx):
    """BONUS archetype. % of messages mentioning placements/exams/deadlines etc.
    Threshold 15%, and needs at least 30 messages so a very quiet person can't win on 2 messages."""
    mine = msgs_of(person, ctx['text'])
    if len(mine) < 30:
        return 0, 15.0
    hits = sum(1 for m in mine if any(k in m['text'].lower() for k in PANDIT_WORDS))
    return pct(hits, len(mine)), 15.0


ARCHETYPES = {
    'THE SPAMMER': score_spammer,
    'THE GROUP MOM': score_group_mom,
    'THE NIGHT OWL': score_night_owl,
    'THE STORYTELLER': score_storyteller,
    'THE DRAMA QUEEN': score_drama_queen,
    'THE GHOST': score_ghost,
    'THE COMEDIAN': score_comedian,
    'THE QUESTION MASTER': score_question_master,
    'THE PLACEMENT PANDIT': score_placement_pandit,
}


def describe(archetype, metric):
    if archetype == 'THE SPAMMER':
        return f"avg {metric:.1f} msgs in a row"
    if archetype == 'THE GROUP MOM':
        return f"{metric:.0f}% msgs with caring words"
    if archetype == 'THE NIGHT OWL':
        return f"{metric:.1f}% msgs between 23h-04h"
    if archetype == 'THE STORYTELLER':
        return f"avg {metric:.1f} words per msg"
    if archetype == 'THE DRAMA QUEEN':
        return f"{metric:.1f}% ALL-CAPS / shouty msgs"
    if archetype == 'THE GHOST':
        return f"silent on {metric:.0f}% of days"
    if archetype == 'THE COMEDIAN':
        return f"{metric:.1f}% msgs with laughs"
    if archetype == 'THE QUESTION MASTER':
        return f"{metric:.1f}% msgs end with '?'"
    return f"{metric:.1f}% msgs about college grind"


def assign_archetypes(ctx, people):
    """Returns {person: (archetype, metric, runner_up)} - exclusive assignment."""
    pairs = []
    for p in people:
        for name, fn in ARCHETYPES.items():
            metric, threshold = fn(p, ctx)
            pairs.append((metric / threshold, p, name, metric))
    pairs.sort(key=lambda t: t[0], reverse=True)

    assigned, used = {}, set()
    for score, p, name, metric in pairs:
        if p not in assigned and name not in used:
            assigned[p] = (name, metric)
            used.add(name)

    result = {}
    for p in people:
        name, metric = assigned[p]
        runner = [(s, n) for s, q, n, _ in pairs if q == p and n != name]
        result[p] = (name, metric, runner[0][1] if runner else '-')
    return result


if HAS_DATA:
    ctx = {'all': messages, 'text': text_messages,
           'active_days': active_days, 'total_days': total_days}
    archetypes = assign_archetypes(ctx, people)
    for p in people:
        name, metric, runner = archetypes[p]
        print(f"{p:<8} -> {name:<21} ({describe(name, metric)})   runner-up: {runner}")

Aman     -> THE NIGHT OWL         (79.8% msgs between 23h-04h)   runner-up: THE SPAMMER
Karan    -> THE STORYTELLER       (avg 57.0 words per msg)   runner-up: THE PLACEMENT PANDIT
Neha     -> THE DRAMA QUEEN       (63.3% ALL-CAPS / shouty msgs)   runner-up: THE SPAMMER
Priya    -> THE GROUP MOM         (64% msgs with caring words)   runner-up: THE QUESTION MASTER
Rahul    -> THE SPAMMER           (avg 4.5 msgs in a row)   runner-up: THE NIGHT OWL
Vikas    -> THE GHOST             (silent on 73% of days)   runner-up: THE COMEDIAN


Feature 8:The Final Report

In [ ]:
WIDTH = 60


def section(title):
    print()
    print(f" {title}")


def print_report(group_name):
    line = '=' * WIDTH
    print(line)
    print(f" GROUPDNA REPORT \u2014 \"{group_name}\"")
    print(f" {total_days} days \u2022 {len(messages):,} messages \u2022 {len(people)} members")
    print(line)
    print(f" Period      : {messages[0]['dt'].strftime('%d %B %Y')} to {messages[-1]['dt'].strftime('%d %B %Y')}")
    print(f" Busiest day : {bd} ({bd_n} messages)")
    print(f" Busiest hour: {bh:02d}:00 - {bh + 1:02d}:00 (avg {bh_n / total_days:.0f} msgs per day)")
    print(f" System msgs : {system_count}   Media: {sum(media_count.values())}   Deleted: {sum(deleted_count.values())}")

    section('MESSAGES PER PERSON')
    top = ranking[0][1]
    for name, n in ranking:
        print(f"  {name:<8} {bar(n, top):<20} {n:>5} ({100 * n / len(messages):4.1f}%)")

    section('ACTIVITY HEATMAP (hour of day, columns 00 to 23)')
    print('           ' + ''.join(f"{h:02d}    " if h % 3 == 0 else '' for h in range(24)))
    for i, p in enumerate(people):
        tag = '  <- ' + archetypes[p][0] if archetypes[p][0] == 'THE NIGHT OWL' else ''
        print(f"  {p:<8} {heatmap_row(heat[i])}{tag}")

    section("THIS GROUP'S FAVOURITE WORDS")
    for w, c in group_top[:5]:
        print(f"  {w:<10} {bar(c, group_top[0][1]):<20} {c}")

    section('RESPONSE PATTERNS')
    valid = {p: v for p, v in avg_gap.items() if v is not None}
    if valid:
        fast = min(valid, key=valid.get)
        slow = max(valid, key=valid.get)
        print(f"  Fastest replier : {fast} (avg {pretty_duration(valid[fast])})")
        print(f"  Slowest replier : {slow} (avg {pretty_duration(valid[slow])})")
    else:
        print('  Not enough back-and-forth to measure.')

    section('LONGEST SILENT STREAKS')
    for p, (n, s, e) in sorted(streaks.items(), key=lambda kv: kv[1][0], reverse=True):
        when = f" ({s.strftime('%d %b')} - {e.strftime('%d %b')})" if n else ''
        print(f"  {p:<8}: {n} day{'s' if n != 1 else ''}{when}")

    section('PERSONALITY ARCHETYPES')
    for p in people:
        name, metric, runner = archetypes[p]
        print(f"  {p:<8} \u2192 {name:<20} ({describe(name, metric)})")

    print()
    print(line)
    print(' Generated by GroupDNA \u2022 Built with Python + NumPy')
    print(line)


if HAS_DATA:
    print_report('Hostel Bois 4ever')
else:
    print('Empty chat - no report to show.')

 GROUPDNA REPORT — "Hostel Bois 4ever"
 60 days • 3,174 messages • 6 members
 Period      : 01 April 2024 to 30 May 2024
 Busiest day : 04 May 2024 (76 messages)
 Busiest hour: 18:00 - 19:00 (avg 4 msgs per day)
 System msgs : 4   Media: 32   Deleted: 15

 MESSAGES PER PERSON
  Rahul    ████████████████████   953 (30.0%)
  Priya    ███████████████        718 (22.6%)
  Neha     █████████████          635 (20.0%)
  Aman     ██████████             490 (15.4%)
  Karan    ███████                354 (11.2%)
  Vikas    █                       24 ( 0.8%)

 ACTIVITY HEATMAP (hour of day, columns 00 to 23)
           00    03    06    09    12    15    18    21    
  Aman     ▒ █ ▒ ▒ █ . . . . . . . . . . . . . . . . . . ▒   <- THE NIGHT OWL
  Karan    . . . . . . . . ░ ░ ▒ ░ █ ▒ █ ▒ ▒ ▒ ▒ █ ▒ ░ . . 
  Neha     . . . . . ░ . . ▒ █ █ ░ ▒ ▒ ░ . ▒ █ █ █ ▒ ░ ░ ░ 
  Priya    . . . . . . . ░ ▒ █ █ █ █ ▒ ▒ ░ ░ ▒ ▒ █ ▒ ░ ░ . 
  Rahul    . . . . . . . . . . . . ▒ ░ ░ ▒ ▒ ░ █ ▒ ░ █ ▒ ▒ 
  Vikas    . . . .

AI disclosure:I used Claude (Anthropic) as a learning aid throughout this project — to help design the parser logic, plan the archetype scoring system, and debug errors.

Reflection

I found the chat parsing part difficult since the WhatsApp file contained not only regular messages but also other types that needed special attention before analyzing.

Next time I will be doing such project, I will first plan out the functions and data structures to use and then proceed with coding. This way, I will make it easier for myself to understand the code and minimize the debugging time.
